# NDgpu — coupled neutronics/thermal transient on GPU (Colab)

How long does a **realistic** coupled transient of the HP-MR microreactor take?

The run is a control-drum manoeuvre from the converged coupled steady state at
2 MWt: the drums withdraw a few degrees over a few seconds, power rises on the
delayed-neutron clock, the fuel heats, and Doppler broadening — interpolated
from the VTB library's own Tfuel branches — pushes back.

**Read this before spending a session on it.** The honest answer to the
headline question is *long*. Measured on this box (CPU, 8 cores) at refine 3
with the real 11-group set, one coupled step costs **~12.8 s**, so a 120 s
transient is ~8.5 hours; refine 6 is several times that. That cost is dominated
by the number of within-step fission-source sweeps, which is a property of the
physics — a near-critical, loosely-coupled, upscattering core — and **not**
something a GPU reduces. A GPU makes each sweep faster; it does not make fewer
sweeps necessary.

So this notebook does **not** attempt a full transient. It measures the one
quantity that transfers — **wall time per neutronics step** — across mesh
refinement, group count and backend, and projects from it. That is the number
you actually need to plan a run, and it is measurable in minutes rather than
hours.

What it covers:

1. **Both backends agree** — a correctness gate before any timing.
2. **CPU vs GPU per step**, across mesh refinement.
3. **What the group count costs** (2-group placeholder vs real 11-group).
4. **Projection**: ms/step x steps for runs you might actually want.
5. **A short real transient** on the fast 2-group set, for the physics plot.


In [ ]:
import os
try:                                        # Colab: upload dist/ndgpu-src.zip
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    get_ipython().run_line_magic("pip", f"install -q {zip_name}")
    try:
        import cupy
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q cupy-cuda12x")
    get_ipython().system("nvidia-smi -L")
except ImportError:                         # local run: ndgpu already importable
    pass

In [ ]:
import os
import time
import numpy as np

from ndgpu.benchmarks.hpmr import build_hpmr2d, build_hpmr3d
from ndgpu.benchmarks.hpmr_thermal import (RATED_POWER_W,
                                           hpmr_angle_for_dollars,
                                           build_hpmr_coupling,
                                           hpmr_drum_ramp, hpmr_endfb8_builtin,
                                           sink_coefficient)
from ndgpu.coupling import coupled_transient

try:
    import cupy
    HAVE_GPU = cupy.cuda.runtime.getDeviceCount() > 0
except Exception:
    HAVE_GPU = False
print("GPU available:", HAVE_GPU)

QUICK = bool(os.environ.get("NDGPU_QUICK"))

TAU = 3.6 / sink_coefficient()
print(f"fuel thermal time constant rho*cp/h = {TAU:.0f} s")


def run(refine, groups="11", nz=0, t_end=2.0, dt=0.05, dt_thermal=0.5,
        device="cpu", drum=None, ramp=4.0, dollars=0.25):
    """One coupled transient; returns (result, seconds, n_active_cells)."""
    mats = hpmr_endfb8_builtin(three_d=nz > 0) if groups == "11" else None
    # Manoeuvres are specified in DOLLARS, not degrees. The drum has almost no
    # worth left above ~150 deg -- the whole travel from 150 to 180 is ~0.2 $ --
    # so the old (150, 153) default inserted 0.05 $ and moved the fuel by
    # hundredths of a kelvin, which looked like a broken thermal coupling.
    if drum is None:
        base = 90.0
        end, pcm = hpmr_angle_for_dollars(base, dollars, refine=refine, nz=nz,
                                          materials=mats, device=device,
                                          with_worth=True)
        drum = (base, end)
    if nz:
        problem = build_hpmr3d(refine=refine, nz=nz, drum_angle_deg=drum[0],
                               absorber="polar", materials=mats)
    else:
        problem = build_hpmr2d(refine=refine, drum_angle_deg=drum[0],
                               absorber="polar", materials=mats)
    ctx = build_hpmr_coupling(problem, device=device)
    # The ramp RATE has to be realistic, not just present. Two ways to get this
    # wrong, both measured at refine 3 against the 12.8 s/step of a full run:
    #   * ramp starting after the probe ends -> unperturbed steps, 4.2 s/step,
    #     3x optimistic (the fixed point starts already converged);
    #   * whole rotation compressed into the probe -> 17.5 s/step, pessimistic.
    # So hold the manoeuvre at its physical duration and probe the first few
    # steps of it. Those are ramp-phase steps, the expensive ones; steps after
    # the drums stop are cheaper, which makes the projection an upper bound.
    pa = hpmr_drum_ramp(problem, angle_from=drum[0], angle_to=drum[1],
                        t_start=0.0, t_ramp=ramp, n_angles=9,
                        refine=refine, nz=nz, materials=mats)
    t0 = time.perf_counter()
    res = coupled_transient(ctx, t_end=t_end, dt=dt, dt_thermal=dt_thermal,
                            problem_at=pa)
    return res, time.perf_counter() - t0, int(np.count_nonzero(problem.active))

## 1. The two backends must agree

Before any timing. A coupled transient has several places a backend difference
could hide — the feedback hook, the device-side power edit, the conduction step —
so the power history is compared end to end.

In [ ]:
cpu_res, cpu_s, ncell = run(refine=3, groups="2", t_end=1.0, device="cpu")
print(f"cpu  P/P0(end) = {cpu_res.power[-1]:.9f}   T_fuel = {cpu_res.mean_temperature[-1]:.6f} K   {cpu_s:.1f} s")

if HAVE_GPU:
    gpu_res, gpu_s, _ = run(refine=3, groups="2", t_end=1.0, device="gpu")
    print(f"gpu  P/P0(end) = {gpu_res.power[-1]:.9f}   T_fuel = {gpu_res.mean_temperature[-1]:.6f} K   {gpu_s:.1f} s")
    dp = np.max(np.abs(gpu_res.power - cpu_res.power))
    dT = abs(gpu_res.mean_temperature[-1] - cpu_res.mean_temperature[-1])
    print(f"\nmax |dP/P0| = {dp:.2e}   |dT| = {dT:.2e} K")
    # float64 on both sides, same algorithm, different reduction order only.
    assert dp < 1e-9 and dT < 1e-6, "backends disagree"
    print("backends agree")

## 2. Cost per neutronics step vs mesh size

`ms/step` is the number that scales. Wall time for a real run is just this times
the step count, and the step count is set by the physics (`t_end / dt`), not by
the machine.

In [ ]:
REFINES = [3, 4] if QUICK else [3, 4, 6]
# Three steps is enough to price a step: the first is the expensive one (cold
# fixed point), later ones are warm-started, and a longer span just multiplies.
T_PROBE, DT = 0.30, 0.05   # 6 ramp-phase steps
rows = []
for refine in REFINES:
    for device in (["cpu", "gpu"] if HAVE_GPU else ["cpu"]):
        res, secs, ncell = run(refine=refine, groups="11", t_end=T_PROBE, dt=DT,
                               dt_thermal=0.5, device=device)
        rows.append((refine, ncell, device, res.steps, 1000 * secs / res.steps,
                     res.steady.seconds, res.power[-1]))
        print(f"refine {refine}  {ncell:7,d} cells  {device:3s}  "
              f"{1000*secs/res.steps:9.0f} ms/step   steady {res.steady.seconds:6.1f} s   "
              f"P/P0 {res.power[-1]:.5f}", flush=True)

if HAVE_GPU:
    print("\nrefine   cells      cpu ms/step   gpu ms/step   speed-up")
    for refine in REFINES:
        c = next(r for r in rows if r[0] == refine and r[2] == "cpu")
        g = next(r for r in rows if r[0] == refine and r[2] == "gpu")
        print(f"{refine:^6d} {c[1]:8,d}   {c[4]:11.0f}   {g[4]:11.0f}   {c[4]/g[4]:7.2f}x")

## 3. What the group count costs

The 2-group set is a placeholder with no `kappaFission`; the 11-group ENDF/B-8
set is the one to quote physics from. The ratio here is what you pay for that.

In [ ]:
for device in (["cpu", "gpu"] if HAVE_GPU else ["cpu"]):
    per = {}
    for groups in ("2", "11"):
        res, secs, ncell = run(refine=4, groups=groups, t_end=0.30, device=device)
        per[groups] = 1000 * secs / res.steps
        print(f"{device:3s}  {groups:>2s} groups  {per[groups]:9.0f} ms/step", flush=True)
    print(f"{device:3s}  11g / 2g = {per['11'] / per['2']:.1f}x\n")

# The ratio is far above the naive 5.5x of group count alone: the 11-group set
# has upscatter and sits near critical, so each step needs many more
# fission-source sweeps, not just more arithmetic per sweep.

## 4. Projection

`ms/step` from section 2, times the step count the physics dictates. Nothing
here is run — that is the point.

**The per-step cost depends on how hard each step is perturbed**, i.e. on the
manoeuvre rate, not only on the mesh. Measured at refine 3 with the 11-group
set, all for the same 3-degree rotation:

| manoeuvre | ms/step |
|---|---|
| drums stationary | ~4,200 |
| 3 deg over 4 s (the default here, and in `examples/hpmr_coupled_transient.py`) | ~2,000 |
| 3 deg over 0.4 s | ~12,800 |
| 3 deg inside 3 steps | ~17,500 |

A faster insertion throws the fixed point further each step and costs more to
re-converge. So quote the table against a stated manoeuvre; it is not a
property of the mesh alone. (Stationary steps look cheap for the opposite
reason — the fixed point starts converged — but they are not what a transient
is made of.)

These are ramp-phase steps; once the drums stop, steps get cheaper, so the
table is an upper bound for a run of the same rate. And note how quickly a
*settled* manoeuvre gets out of reach: the fuel's thermal constant is 268 s, so
watching the excursion actually be arrested takes many minutes of reactor
time.

In [ ]:
def pretty(seconds):
    if seconds < 120:   return f"{seconds:.0f} s"
    if seconds < 7200:  return f"{seconds/60:.0f} min"
    if seconds < 172800: return f"{seconds/3600:.1f} h"
    return f"{seconds/86400:.1f} days"

print(f"{'case':28s} {'ms/step':>9}   " + "".join(f"{t:>12s}" for t in
      ("2 s", "10 s", "120 s", "600 s")))
for refine, ncell, device, steps, ms, steady, _ in rows:
    line = f"refine {refine} {device:3s} {ncell:7,d} cells {ms:9.0f}   "
    for t_end in (2.0, 10.0, 120.0, 600.0):
        line += f"{pretty(ms * (t_end / DT) / 1000):>12s}"
    print(line)

print("\nThe steady state is charged once per run, not per step:",
      f"{rows[0][5]:.0f} s at refine {rows[0][0]}.")

## 5. A short transient, for the shape

Run on the **2-group** set, where a step is ~50 ms rather than ~13 s, so the
power and temperature histories are actually plottable. The shapes are right;
the numbers are illustrative — the 2-group set is a placeholder with no
kappaFission. Quote physics from the 11-group runs above.

In [ ]:
DEVICE = "gpu" if HAVE_GPU else "cpu"
T_RUN = 2.0 if QUICK else 20.0
res, secs, ncell = run(refine=4, groups="2", t_end=T_RUN, dt=0.05,
                       dt_thermal=0.5, device=DEVICE)
print(res)
print(f"\n  k_eff at the initial coupled state : {res.k0:.6f}")
print(f"  peak P/P0                          : {res.power.max():.4f}")
print(f"  fuel temperature rise              : "
      f"{res.mean_temperature[-1] - res.mean_temperature[0]:+.2f} K mean")
print(f"  wall                               : {secs:.1f} s on {res.device} "
      f"({1000*secs/res.steps:.0f} ms/step)")

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].plot(res.times, res.power, lw=2)
    ax[0].set_xlabel("time, s"); ax[0].set_ylabel("P / P0")
    ax[0].set_title("power"); ax[0].grid(alpha=.3)
    ax[1].plot(res.times, res.mean_temperature, lw=2, label="fuel mean")
    ax[1].plot(res.times, res.peak_temperature, lw=2, label="fuel peak")
    ax[1].set_xlabel("time, s"); ax[1].set_ylabel("temperature, K")
    ax[1].set_title("fuel temperature"); ax[1].legend(); ax[1].grid(alpha=.3)
    fig.tight_layout()
except ImportError:
    pass

## How to read the numbers

- **ms/step is the currency**, and the step count is set by the physics. Both
  are in the projection table; nothing else is needed to plan a run.
- **The GPU speeds up sweeps, not sweep counts.** Expect it to help roughly in
  proportion to cells x groups on the linear algebra, and to help least where
  the bill is dominated by *how many* fission-source sweeps a near-critical
  step needs. On small meshes the CPU can win outright — the work is
  launch-bound.
- **The within-step accelerator matters more than the hardware here.**
  Whole-core rebalance (`rebalance=True`, Anderson off) is ~1.5x the previous
  Anderson default end-to-end on the 11-group core, at a better-converged
  answer; `ndgpu.coupling.coupled_transient` selects it automatically above two
  groups. Rebalance and Anderson do **not** combine.
- **Anderson's convergence test is optimistic on this map** — at the same
  nominal tolerance it stopped 0.58% short of the converged power on the
  11-group core. If you use it here, tighten `tol_step`.
- **What is not modelled:** the mesh does not expand, the heat pipes are a
  volumetric conductance to a fixed evaporator temperature rather than a
  wick-limited device, and the moderator is assumed to sit at the fuel
  temperature (the library branch is read along Tfuel = Tmod).